# Team MLB · SVM 실험

데이터팀이 만든 특징 CSV(`features_*.csv`, 정보 열 6개 + 특징 110개)를 읽어서 RBF 커널 SVM을 학습한다.
분할·교차검증·지표·결과 파일 형식은 Logistic Regression 노트북(`MLB_LogisticRegression_v3`)과 같게 맞췄다.

| 순서 | 내용 |
| --- | --- |
| 0 | 설정 · 드라이브 연결 |
| 1 | CSV 로드 · 점검 |
| 2 | 학습/최종평가 분할 (`split_seed42.csv` 공유) |
| 3 | 공통 함수: 전처리(log → 상수 제거 → 표준화) + SVM, 5-fold로 C·gamma 선택 |
| 4 | 실험 ① Closed-world: 특징 구성 A(g1+g6) vs B(전체) + 튜닝 전 비교 |
| 4-추가 | 누적 흐름 50개 vs 100개 비교 (D, E) |
| 5 | 실험 ② Open-world 이진 (mon 1 / unmon -1) |
| 5-추가 | 클래스 가중치 균형 비교 |
| 6 | 실험 ③ Open-world 다중 (96 클래스) |
| 7 | 초반 제거 비교 trim0/10/20 (Open-world 이진) |
| 7-추가 | 초반 방향 그룹(g4) 제외 비교 (수집 흔적 점검) |
| 8 | 오류 분석 (사이트별 정확도, 혼동 쌍) |
| 9 | 특징 그룹 제거 실험 (Closed-world, 이진) |
| 10 | 결과 저장 |

## 0. 설정

In [1]:
FEATURE_DIR = "/content/drive/MyDrive/MLB-Project/feature"
RESULT_DIR  = "/content/drive/MyDrive/MLB-Project/results/svm"
SPLIT_PATH  = "/content/drive/MyDrive/MLB-Project/results/split_seed42.csv"   # LR과 같은 분할 파일
UNMON_NAME  = "unmon10k"
TRIM        = 0
TRIMS_TO_COMPARE = [0, 10, 20]

SEED     = 42
N_FOLDS  = 5
N_JOBS   = -1

# C·gamma 탐색: log2 단위. 1차는 2칸 간격, 최적값이 경계면 범위를 넓히고, 마지막에 1칸 간격으로 주변을 세밀 탐색
LOG2_C_COARSE = [1, 3, 5, 7, 9, 11]
LOG2_G_COARSE = [-11, -9, -7, -5, -3, -1]

RUN_ABLATION = True    # 9번 그룹 제거 실험

In [2]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULT_DIR, exist_ok=True)
print("FEATURE_DIR:", FEATURE_DIR)
print("RESULT_DIR :", RESULT_DIR)
print("SPLIT_PATH :", SPLIT_PATH)

FEATURE_DIR: /content/drive/MyDrive/MLB-Project/feature
RESULT_DIR : /content/drive/MyDrive/MLB-Project/results/svm
SPLIT_PATH : /content/drive/MyDrive/MLB-Project/results/split_seed42.csv


In [3]:
import time, itertools
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. CSV 로드 · 점검

- 열은 정보 6개 + 특징 110개여야 한다. NaN·무한대가 있으면 데이터팀에 알린다.
- `g5_iat_pos_max`와 `g5_iat_all_max`는 모든 기록에서 값이 같다(데이터팀 문서 8-4). SVM에서는 같은 정보가 거리 계산에 두 번 들어가므로 `g5_iat_pos_max`를 뺀다. **SVM의 "전체 특징"은 109개**다.
- log 변환 대상: 0 이상이고 분포가 한쪽으로 긴 g1~g5 특징. 시간 간격 사분위수는 왜도가 100을 넘는다. g6 누적값은 음수가 있어 제외한다.

In [4]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]
DUP_DROP  = ["g5_iat_pos_max"]

def load_features(kind, trim):
    path = os.path.join(FEATURE_DIR, f"features_{kind}_trim{trim}.csv")
    df = pd.read_csv(path)
    feat = [c for c in df.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")]
    assert list(df.columns[:6]) == INFO_COLS, f"정보 열 순서가 다름: {path}"
    assert len(feat) == 110, f"특징 수 {len(feat)}개: {path}"
    X = df[feat].to_numpy(dtype=float)
    print(f"{os.path.basename(path)}: {df.shape}, NaN {int(np.isnan(X).sum())}, inf {int(np.isinf(X).sum())}")
    assert (df[[c for c in feat if not c.startswith("g6_")]] >= 0).all().all(), "g1~g5에 음수가 있음 → log 변환 규칙 확인"
    return df

mon, unmon = load_features("mon", TRIM), load_features(UNMON_NAME, TRIM)
FEATS_ALL = [c for c in mon.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_") and c not in DUP_DROP]
GROUPS = {g: [c for c in FEATS_ALL if c.startswith(g + "_")] for g in ["g1", "g2", "g3", "g4", "g5", "g6"]}
LOG_COLS = set(c for c in FEATS_ALL if not c.startswith("g6_"))
print("SVM 전체 특징:", len(FEATS_ALL), {g: len(v) for g, v in GROUPS.items()})

skew = mon[FEATS_ALL].skew().sort_values(ascending=False)
print("왜도 상위 5개 (log 변환 전):"); print(skew.head().round(1).to_string())
const_mon = [c for c in FEATS_ALL if mon[c].nunique() == 1]
print("mon trim0에서 상수인 특징:", const_mon, "→ 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거")

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
SVM 전체 특징: 109 {'g1': 3, 'g2': 17, 'g3': 6, 'g4': 3, 'g5': 30, 'g6': 50}
왜도 상위 5개 (log 변환 전):
g5_iat_all_q50    137.8
g5_iat_in_q50     101.3
g5_iat_out_q25     22.3
g5_iat_out_q75     18.5
g5_iat_in_q75      17.2
mon trim0에서 상수인 특징: ['g5_iat_all_q25', 'g5_iat_in_q25', 'g6_cumul_00'] → 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거


## 2. 학습 / 최종 평가 분할 (모델팀 공유)

LR 노트북이 만든 `split_seed42.csv`를 그대로 읽는다. `record_id` 기준이라 trim0/10/20 모두 같은 분할이 적용된다.
최종 평가 기록은 특징 구성·C·gamma 선택에 쓰지 않는다.

In [5]:
split = pd.read_csv(SPLIT_PATH)
SPLIT = dict(zip(split.record_id, split.split))

def attach_split(df):
    s = df.record_id.map(SPLIT)
    assert s.notna().all(), "분할 파일에 없는 record_id가 있음"
    df = df.copy(); df["split"] = s.to_numpy()
    return df

mon, unmon = attach_split(mon), attach_split(unmon)
_all = pd.concat([mon, unmon], ignore_index=True)
print(pd.crosstab(_all.dataset, _all.split))

split     test  train
dataset              
mon       3800  15200
unmon10k  2000   8000


## 3. 공통 함수

**파이프라인** (5-fold의 각 학습 구간에서만 적합해 정보 유출을 막는다)
1. g1~g5 특징에 `log1p` 적용 (g6 누적값은 그대로)
2. `VarianceThreshold`: 학습 구간에서 상수인 특징 제거 (trim·데이터마다 상수 특징이 다르다)
3. `StandardScaler`
4. `SVC(kernel="rbf")`, 다중 분류는 one-vs-one

**C·gamma 탐색** (`tune`)
- 1차: log2 단위 2칸 간격 격자를 5-fold로 평가한다.
- 최적값이 탐색 범위의 끝이면 그 방향으로 2칸씩 넓혀 다시 평가한다(최대 2번).
- 2차: 최적값 주변을 1칸 간격으로 세밀하게 평가한다.
- `fixed=(log2C, log2gamma)`를 주면 탐색 없이 그 설정의 5-fold 점수만 구한다(같은 설정으로 조건을 비교하는 7·9번 실험).

Open-world 이진의 ROC·PR 곡선은 `decision_function` 점수로 계산한다. 확률 추정(`probability=True`)은 학습 시간이 크게 늘고 이진 곡선에는 필요하지 않아 쓰지 않는다. 판정 기준은 LR(확률 0.5)과 맞춰 SVM 기본 기준(점수 0)을 쓴다.

In [6]:
def safe_log1p(X):
    return np.log1p(np.clip(X, 0, None))

def make_svm(feats, C=1.0, gamma="scale", class_weight=None):
    log_cols = [c for c in feats if c in LOG_COLS]
    pre = ColumnTransformer([("log", FunctionTransformer(safe_log1p, feature_names_out="one-to-one"), log_cols)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("var", VarianceThreshold()), ("sc", StandardScaler()),
                     ("svc", SVC(kernel="rbf", C=C, gamma=gamma, class_weight=class_weight, cache_size=1000))])

CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
CV_TABLES = {}

def _grid_eval(X, y, pairs, class_weight):
    grid = [{"svc__C": [2.0 ** c], "svc__gamma": [2.0 ** g]} for c, g in pairs]
    gs = GridSearchCV(make_svm(list(X.columns), class_weight=class_weight), grid, cv=CV,
                      scoring="accuracy", n_jobs=N_JOBS, refit=False)
    gs.fit(X, y)
    out = {}
    for p, m, s, ft in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"],
                           gs.cv_results_["std_test_score"], gs.cv_results_["mean_fit_time"]):
        out[(int(round(np.log2(p["svc__C"]))), int(round(np.log2(p["svc__gamma"]))))] = (m, s, ft)
    return out

def tune(X, y, name, log2C=None, log2G=None, class_weight=None, fixed=None, max_extend=2):
    t0 = time.time()
    tried = {}
    def run(pairs):
        new = [p for p in pairs if p not in tried]
        if new:
            tried.update(_grid_eval(X, y, new, class_weight))
    best = lambda: max(tried, key=lambda k: tried[k][0])
    if fixed is not None:
        run([tuple(fixed)])
    else:
        log2C = LOG2_C_COARSE if log2C is None else log2C
        log2G = LOG2_G_COARSE if log2G is None else log2G
        run(list(itertools.product(log2C, log2G)))
        for _ in range(max_extend):                       # 경계에 걸리면 범위 확장
            c, g = best()
            cs = sorted({k[0] for k in tried}); gs_ = sorted({k[1] for k in tried})
            ext = []
            if c == cs[0]:  ext += [(c - 2, gg) for gg in gs_]
            if c == cs[-1]: ext += [(c + 2, gg) for gg in gs_]
            if g == gs_[0]:  ext += [(cc, g - 2) for cc in cs]
            if g == gs_[-1]: ext += [(cc, g + 2) for cc in cs]
            if not ext: break
            print(f"  [{name}] 최적값 (log2C={c}, log2γ={g})이 경계 → 범위 확장")
            run(ext)
        c, g = best()
        run([(c + dc, g + dg) for dc in (-1, 0, 1) for dg in (-1, 0, 1)])   # 1칸 간격 세밀 탐색
    t_search = time.time() - t0
    c, g = best()
    tab = pd.DataFrame([dict(log2C=k[0], log2gamma=k[1], cv_acc_mean=v[0], cv_acc_std=v[1], fit_time_per_fold=v[2])
                        for k, v in tried.items()]).sort_values("cv_acc_mean", ascending=False).reset_index(drop=True)
    CV_TABLES[name] = tab.assign(experiment=name)
    info = dict(log2C=c, log2gamma=g, best_C=2.0 ** c, best_gamma=2.0 ** g,
                cv_acc=tried[(c, g)][0], cv_std=tried[(c, g)][1], n_evaluated=len(tried), search_time_s=t_search)
    print(f"[{name}] best log2C={c}, log2γ={g} | CV acc = {info['cv_acc']:.4f} ± {info['cv_std']:.4f} "
          f"| 평가한 조합 {len(tried)}개 | 탐색 {t_search:.0f}s")
    if fixed is None:
        print(tab.head(8).round(4).to_string())
    return info

def refit(X, y, info, class_weight=None):
    model = make_svm(list(X.columns), C=info["best_C"], gamma=info["best_gamma"], class_weight=class_weight)
    t0 = time.time(); model.fit(X, y); info["refit_time_s"] = time.time() - t0
    info["n_support"] = int(model[-1].n_support_.sum())
    return model

def timed_predict(model, X):
    t0 = time.time(); p = model.predict(X); return p, time.time() - t0

RESULTS, PREDS = [], {}

## 4. 실험 ① Closed-world: 관심 사이트 95개 식별

특징 구성 두 가지를 비교한다. 선택은 **5-fold CV 정확도**로만 하고 최종 평가 데이터는 보지 않는다.

| 조건 | 특징 | 개수 | 근거 |
| --- | --- | --- | --- |
| A | g1 + g6 | 53 | 제안서의 SVM 주 입력 (CUMUL 연구 구성) |
| B | 전체 | 109 | 다른 모델과 같은 조건 |

튜닝 전후 비교: 기본 설정(C=1, gamma="scale")의 결과도 남긴다.

In [7]:
FEATS_A = GROUPS["g1"] + GROUPS["g6"]
FEATS_B = FEATS_ALL

def run_closed(mon, feats, name, **tune_kw):
    tr, te = mon[mon.split == "train"], mon[mon.split == "test"]
    info = tune(tr[feats], tr.label, name, **tune_kw)
    model = refit(tr[feats], tr.label, info)
    pred, t_pred = timed_predict(model, te[feats])
    row = dict(experiment=name, trim=int(mon.trim.iloc[0]), n_features=len(feats), **info,
               test_acc=accuracy_score(te.label, pred), test_macro_f1=f1_score(te.label, pred, average="macro"),
               predict_time_s=t_pred)
    PREDS[name] = te[["record_id", "label"]].assign(pred=pred)
    return row, model

closed_rows = {}
for tag, feats in [("A", FEATS_A), ("B", FEATS_B)]:
    tr = mon[mon.split == "train"]
    closed_rows[tag] = tune(tr[feats], tr.label, f"closed_cv_{tag}")

choice = max(closed_rows, key=lambda k: closed_rows[k]["cv_acc"])
FEATS_MAIN = {"A": FEATS_A, "B": FEATS_B}[choice]
print(f"\n→ CV 기준으로 조건 {choice}를 선택 (A {closed_rows['A']['cv_acc']:.4f} vs B {closed_rows['B']['cv_acc']:.4f})")

  [closed_cv_A] 최적값 (log2C=11, log2γ=-3)이 경계 → 범위 확장
  [closed_cv_A] 최적값 (log2C=13, log2γ=-3)이 경계 → 범위 확장
[closed_cv_A] best log2C=14, log2γ=-2 | CV acc = 0.8523 ± 0.0057 | 평가한 조합 56개 | 탐색 1351s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0     14         -2       0.8523      0.0057             3.5258
1     16         -2       0.8522      0.0055             3.4707
2     15         -2       0.8521      0.0055             3.4633
3     15         -3       0.8498      0.0042             3.9528
4     16         -3       0.8495      0.0038             3.9811
5     14         -3       0.8494      0.0047             3.9259
6     13         -3       0.8492      0.0044             3.9789
7     16         -4       0.8491      0.0026             5.0422
[closed_cv_B] best log2C=6, log2γ=-6 | CV acc = 0.8428 ± 0.0051 | 평가한 조합 44개 | 탐색 1340s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      6         -6       0.8428      0.0051             4.1019
1      5     

In [8]:
# 선택한 조건으로 전체 학습 데이터 재학습 → 최종 평가 (탐색은 위에서 끝났으므로 fixed로 다시 점수만 확인)
best = closed_rows[choice]
row, model_closed = run_closed(mon, FEATS_MAIN, "closed_world", fixed=(best["log2C"], best["log2gamma"]))
RESULTS.append(row)
BEST_CLOSED = (row["log2C"], row["log2gamma"])
print(f"→ test acc {row['test_acc']:.4f} | macro-F1 {row['test_macro_f1']:.4f} | 서포트 벡터 {row['n_support']}개")

# 선택하지 않은 조건도 참고용으로 같은 방식으로 기록
other = "B" if choice == "A" else "A"
row_o, _ = run_closed(mon, {"A": FEATS_A, "B": FEATS_B}[other], f"closed_world_cond{other}",
                      fixed=(closed_rows[other]["log2C"], closed_rows[other]["log2gamma"]))
RESULTS.append(row_o)
print(f"[참고] 조건 {other} test acc {row_o['test_acc']:.4f}")

# 튜닝 전(기본 설정 C=1, gamma='scale')
tr, te = mon[mon.split == "train"], mon[mon.split == "test"]
m0 = make_svm(FEATS_MAIN)
cvs = cross_validate(m0, tr[FEATS_MAIN], tr.label, cv=CV, scoring="accuracy", n_jobs=N_JOBS)
m0.fit(tr[FEATS_MAIN], tr.label)
p0 = m0.predict(te[FEATS_MAIN])
RESULTS.append(dict(experiment="closed_world_default", trim=0, n_features=len(FEATS_MAIN), best_C=1.0, best_gamma="scale",
                    cv_acc=cvs["test_score"].mean(), cv_std=cvs["test_score"].std(),
                    test_acc=accuracy_score(te.label, p0), test_macro_f1=f1_score(te.label, p0, average="macro")))
print(f"[튜닝 전 C=1, gamma=scale] CV acc {cvs['test_score'].mean():.4f} ± {cvs['test_score'].std():.4f} | test acc {accuracy_score(te.label, p0):.4f}")

[closed_world] best log2C=14, log2γ=-2 | CV acc = 0.8523 ± 0.0057 | 평가한 조합 1개 | 탐색 23s
→ test acc 0.8650 | macro-F1 0.8646 | 서포트 벡터 10359개
[closed_world_condB] best log2C=6, log2γ=-6 | CV acc = 0.8428 ± 0.0051 | 평가한 조합 1개 | 탐색 28s
[참고] 조건 B test acc 0.8534
[튜닝 전 C=1, gamma=scale] CV acc 0.2955 ± 0.0053 | test acc 0.3184


### 4-추가. 누적 흐름 50개 vs 100개

`cumul100_mon_trim0.csv`(monitored trim0 전용)를 `record_id`로 붙여 g6 50개를 100개로 바꾼다. 50개 조건(A, B)에서 찾은 C·gamma 주변(±1칸)만 다시 탐색한다.

| 조건 | 특징 | 개수 | 비교 대상 |
| --- | --- | --- | --- |
| D | g1 + 누적 100 | 103 | A |
| E | g1~g5 + 누적 100 | 159 | B |

In [9]:
c100 = pd.read_csv(os.path.join(FEATURE_DIR, "cumul100_mon_trim0.csv"))
assert (c100.record_id.values == mon.record_id.values).all()
C100 = [c for c in c100.columns if c != "record_id"]
mon100 = mon.merge(c100, on="record_id")
FEATS_D = GROUPS["g1"] + C100
FEATS_E = [c for c in FEATS_ALL if not c.startswith("g6_")] + C100

cumul_rows = []
for tag, feats, base in [("D", FEATS_D, "A"), ("E", FEATS_E, "B")]:
    b = closed_rows[base]
    tr = mon100[mon100.split == "train"]
    info = tune(tr[feats], tr.label, f"closed_cv_{tag}",
                log2C=[b["log2C"] - 1, b["log2C"], b["log2C"] + 1], log2G=[b["log2gamma"] - 1, b["log2gamma"], b["log2gamma"] + 1])
    cumul_rows.append(dict(condition=tag, compare_to=base, n_features=len(feats), cv_acc=info["cv_acc"], cv_std=info["cv_std"],
                           base_cv_acc=closed_rows[base]["cv_acc"], log2C=info["log2C"], log2gamma=info["log2gamma"]))
cumul_table = pd.DataFrame(cumul_rows)
cumul_table["delta"] = cumul_table.cv_acc - cumul_table.base_cv_acc
print(cumul_table.round(4).to_string())
cumul_table.to_csv(os.path.join(RESULT_DIR, "svm_cumul50_vs_100.csv"), index=False)

  [closed_cv_D] 최적값 (log2C=14, log2γ=-3)이 경계 → 범위 확장
[closed_cv_D] best log2C=14, log2γ=-3 | CV acc = 0.8673 ± 0.0060 | 평가한 조합 15개 | 탐색 338s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0     14         -3       0.8673      0.0060             3.9824
1     15         -3       0.8672      0.0059             4.1453
2     13         -3       0.8668      0.0060             4.1786
3     15         -4       0.8651      0.0061             4.3201
4     14         -4       0.8651      0.0065             4.4811
5     13         -4       0.8646      0.0064             4.4851
6     15         -2       0.8640      0.0062             4.2203
7     13         -2       0.8639      0.0065             4.0444
  [closed_cv_E] 최적값 (log2C=5, log2γ=-6)이 경계 → 범위 확장
[closed_cv_E] best log2C=5, log2γ=-6 | CV acc = 0.8524 ± 0.0080 | 평가한 조합 15개 | 탐색 444s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      5         -6       0.8524      0.0080             4.7878
1      6        

## 5. 실험 ② Open-world 이진: 관심 사이트인가?

- 데이터: mon(1) + unmon(-1), 특징은 4번에서 고른 구성
- C·gamma는 이진 문제에서 다시 탐색한다.
- 지표는 LR과 같다: 정확도(+다수 범주 기준선), mon→mon(TPR), unmon→unmon(TNR), precision, FPR, ROC-AUC, PR-AUC
- 예측 파일에 `decision_function` 점수(`score`)를 저장해 기준값을 바꿔 보거나 곡선을 그릴 수 있게 한다.

In [10]:
def binary_view(mon, unmon):
    df = pd.concat([mon, unmon], ignore_index=True)
    df["y"] = np.where(df.dataset == "mon", 1, -1)
    return df

def openworld_metrics(y_true_bin, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[-1, 1]).ravel()
    return dict(mon_as_mon=tp / (tp + fn), unmon_as_unmon=tn / (tn + fp),
                precision=tp / (tp + fp) if tp + fp else 0.0, fpr=fp / (fp + tn), TP=tp, FN=fn, TN=tn, FP=fp)

def run_open_binary(mon, unmon, feats, name, class_weight=None, **tune_kw):
    df = binary_view(mon, unmon)
    tr, te = df[df.split == "train"], df[df.split == "test"]
    info = tune(tr[feats], tr.y, name, class_weight=class_weight, **tune_kw)
    model = refit(tr[feats], tr.y, info, class_weight=class_weight)
    pred, t_pred = timed_predict(model, te[feats])
    score = model.decision_function(te[feats])          # 클수록 mon
    m = openworld_metrics(te.y, pred)
    row = dict(experiment=name, trim=int(df.trim.iloc[0]), n_features=len(feats), **info,
               test_acc=accuracy_score(te.y, pred), majority_baseline=max((te.y == 1).mean(), (te.y == -1).mean()),
               roc_auc=roc_auc_score(te.y, score), pr_auc=average_precision_score(te.y, score, pos_label=1),
               predict_time_s=t_pred, **m)
    print(f"→ test acc {row['test_acc']:.4f} (다수 범주 {row['majority_baseline']:.4f}) | mon→mon {m['mon_as_mon']:.4f} | "
          f"unmon→unmon {m['unmon_as_unmon']:.4f} | precision {m['precision']:.4f} | ROC-AUC {row['roc_auc']:.4f}")
    PREDS[f"{name}_trim{row['trim']}"] = te[["record_id", "dataset", "y"]].assign(pred=pred, score=score)
    return row, model

# 탐색 범위: 처음에는 log2C 1~9, log2γ (Closed 최적 ±2)로 넓게 탐색했으나 최적값이 계속 큰 gamma 쪽 경계로 이동해
# (log2C 9, log2γ 0) → (7, 2)) 2시간 안에 끝나지 않았다. 그 결과를 바탕으로 (7, 2) 주변으로 범위를 좁혔다.
row, model_binary = run_open_binary(mon, unmon, FEATS_MAIN, "open_binary",
                                    log2C=[5, 7, 9], log2G=[1, 2, 3], max_extend=1)
RESULTS.append(row)
BEST_BINARY = (row["log2C"], row["log2gamma"])

/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
  [open_binary] 최적값 (log2C=7, log2γ=1)이 경계 → 범위 확장
[open_binary] best log2C=7, log2γ=1 | CV acc = 0.8322 ± 0.0057 | 평가한 조합 19개 | 탐색 1856s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      7          1       0.8322      0.0057            36.7463
1      7          2       0.8319      0.0054            31.1682
2      9          3       0.8317      0.0062            34.7775
3      8          2       0.8316      0.0049            32.6461
4      9          2       0.8316      0.0064            36.6331
5      7          3       0.8315      0.0060            35.3927
6      8          1       0.8314      0.0062            45.6542
7      9          1       0.8309      0.0058            59.6053
→ test acc 0.8428 (다수 범주 0.6552) | mon

### 5-추가. 클래스 가중치 균형 비교

LR과 같은 이유(학습 데이터 mon:unmon ≈ 2:1)로 `class_weight="balanced"`를 비교한다. 탐색 비용을 줄이기 위해 5번의 C·gamma를 그대로 쓴다(LR은 C를 다시 골랐다는 차이가 있다).
ROC-AUC가 비슷하면 구분 능력이 아니라 판정 기준만 옮겨진 것이다.

In [11]:
row_bal, _ = run_open_binary(mon, unmon, FEATS_MAIN, "open_binary_balanced", class_weight="balanced", fixed=BEST_BINARY)
RESULTS.append(row_bal)
print(pd.DataFrame([row, row_bal])[["experiment", "test_acc", "mon_as_mon", "unmon_as_unmon", "precision", "fpr", "roc_auc"]].round(4).to_string())

[open_binary_balanced] best log2C=7, log2γ=1 | CV acc = 0.8300 ± 0.0067 | 평가한 조합 1개 | 탐색 115s
→ test acc 0.8405 (다수 범주 0.6552) | mon→mon 0.8858 | unmon→unmon 0.7545 | precision 0.8727 | ROC-AUC 0.8861
             experiment  test_acc  mon_as_mon  unmon_as_unmon  precision     fpr  roc_auc
0           open_binary    0.8428      0.9053          0.7240     0.8617  0.2760   0.8868
1  open_binary_balanced    0.8405      0.8858          0.7545     0.8727  0.2455   0.8861


## 6. 실험 ③ Open-world 다중: 95개 + unmon(-1) = 96 클래스

- LR과 같이 -1을 하나의 클래스로 학습하고 예측 결과를 그대로 쓴다(별도 확률 기준값 없음).
- 탐색은 Closed-world 최적값 주변으로 좁힌다(96 클래스 학습이 가장 오래 걸린다).
- 지표: 96 클래스 정확도, mon→mon / unmon→unmon(관심 여부만), mon 사이트 정답 비율(사이트 번호까지)

In [12]:
def run_open_multi(mon, unmon, feats, name, **tune_kw):
    df = pd.concat([mon, unmon], ignore_index=True)
    tr, te = df[df.split == "train"], df[df.split == "test"]
    info = tune(tr[feats], tr.label, name, **tune_kw)
    model = refit(tr[feats], tr.label, info)
    pred, t_pred = timed_predict(model, te[feats])
    y = te.label.to_numpy()
    m = openworld_metrics(np.where(y >= 0, 1, -1), np.where(pred >= 0, 1, -1))
    is_mon = y >= 0
    row = dict(experiment=name, trim=int(df.trim.iloc[0]), n_features=len(feats), **info,
               test_acc=accuracy_score(y, pred), mon_site_exact=(pred[is_mon] == y[is_mon]).mean(),
               test_macro_f1=f1_score(y, pred, average="macro"), predict_time_s=t_pred, **m)
    print(f"→ test acc {row['test_acc']:.4f} | mon→mon {m['mon_as_mon']:.4f} | unmon→unmon {m['unmon_as_unmon']:.4f} | "
          f"mon 사이트 정답 {row['mon_site_exact']:.4f}")
    PREDS[name] = te[["record_id", "dataset", "label"]].assign(pred=pred)
    return row, model

c0, g0 = BEST_CLOSED
row, model_multi = run_open_multi(mon, unmon, FEATS_MAIN, "open_multi", log2C=[c0 - 2, c0, c0 + 2], log2G=[g0 - 1, g0, g0 + 1])
RESULTS.append(row)

  [open_multi] 최적값 (log2C=12, log2γ=-3)이 경계 → 범위 확장
[open_multi] best log2C=12, log2γ=-3 | CV acc = 0.8067 ± 0.0045 | 평가한 조합 22개 | 탐색 2064s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0     12         -3       0.8067      0.0045            23.8904
1     11         -2       0.8060      0.0014            18.7806
2     10         -2       0.8058      0.0031            17.8951
3     10         -1       0.8055      0.0061            15.7753
4     13         -3       0.8053      0.0039            27.1822
5     12         -2       0.8053      0.0021            18.9532
6     14         -3       0.8052      0.0041            29.7931
7     13         -2       0.8052      0.0024            19.5914
→ test acc 0.8155 | mon→mon 0.8353 | unmon→unmon 0.8655 | mon 사이트 정답 0.7892


## 7. 초반 제거 비교 (Open-world 이진)

원본·앞 10개 제거·앞 20개 제거를 같은 분할, 같은 C·gamma(5번 결과)로 비교한다.

> **해석 주의 (LR 노트북 7-추가)**: 원본 7번 패킷 방향 하나로 mon/unmon이 0.9999 갈린다. trim0에서는 `g6_cumul_00`이 상수라 이 정보가 특징에 거의 남지 않지만, trim10에서는 원래 10번 패킷 방향이 `g6_cumul_00`에 그대로 들어간다. **trim10의 성능 상승은 수집 흔적이 드러난 결과이며 성능 향상으로 해석하지 않는다.**

In [13]:
trim_rows = []
for t in TRIMS_TO_COMPARE:
    m_t, u_t = attach_split(load_features("mon", t)), attach_split(load_features(UNMON_NAME, t))
    r, _ = run_open_binary(m_t, u_t, FEATS_MAIN, "open_binary_trimcmp", fixed=BEST_BINARY)
    trim_rows.append(r)
trim_table = pd.DataFrame(trim_rows)[["trim", "cv_acc", "cv_std", "test_acc", "roc_auc", "mon_as_mon", "unmon_as_unmon",
                                      "precision", "TP", "TN", "FP", "FN"]]
print(trim_table.round(4).to_string())
RESULTS.extend(trim_rows)

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best log2C=7, log2γ=1 | CV acc = 0.8322 ± 0.0057 | 평가한 조합 1개 | 탐색 111s
→ test acc 0.8428 (다수 범주 0.6552) | mon→mon 0.9053 | unmon→unmon 0.7240 | precision 0.8617 | ROC-AUC 0.8868
features_mon_trim10.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim10.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best log2C=7, log2γ=1 | CV acc = 0.8916 ± 0.0033 | 평가한 조합 1개 | 탐색 68s
→ test acc 0.9012 (다수 범주 0.6552) | mon→mon 0.9426 | unmon→unmon 0.8225 | precision 0.9098 | ROC-AUC 0.9321
features_mon_trim20.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim20.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best log2C=7, log2γ=1 | CV acc = 0.8255 ± 0.0068 | 평가한 조합 1개 | 탐색 106s
→ test acc 0.8374 (다수 범주 0.6552) | mon→mon 0.8947 | unmon→unmon 0.7285 | precision 0.8623 | ROC-AUC 0.8845
   trim  cv_acc  cv_std  test_acc  roc_auc  mon_as_mon  unmon_as_unmon  prec

### 7-추가. 초반 방향 그룹(g4) 제외 비교

원본 8~13번 패킷의 송신 여부는 mon(4/19,000)과 unmon(8,625/10,000)이 크게 다르고, 이 차이는 g4의 "처음 30개 중 송신 비율"에 직접 들어간다. trim0 이진에서 g4를 뺐을 때의 변화로 초반 정보 의존도를 확인한다. g6 앞부분에도 같은 정보가 일부 섞이므로, g4 제외만으로 흔적이 제거됐다고 결론 내리지 않는다.

In [14]:
feats_no_g4 = [c for c in FEATS_MAIN if not c.startswith("g4_")]
if len(feats_no_g4) == len(FEATS_MAIN):
    print("선택한 특징 구성에 g4가 없어 건너뜀")
else:
    row_ng4, _ = run_open_binary(mon, unmon, feats_no_g4, "open_binary_no_g4", fixed=BEST_BINARY)
    RESULTS.append(row_ng4)
    base_b = next(r for r in RESULTS if r["experiment"] == "open_binary")
    print(pd.DataFrame([base_b, row_ng4])[["experiment", "n_features", "cv_acc", "test_acc", "mon_as_mon", "unmon_as_unmon", "roc_auc"]].round(4).to_string())

선택한 특징 구성에 g4가 없어 건너뜀


## 8. 오류 분석 (Closed-world)

최종 평가 기록(사이트당 40개)에서 사이트별 정확도와 자주 혼동한 (정답 → 예측) 쌍을 본다. 사이트당 40개라 한두 개 차이는 우연일 수 있다.

In [15]:
pc = PREDS["closed_world"]
site_acc = (pc.label == pc.pred).groupby(pc.label).mean().rename("test_acc")
print("[Closed-world] 정확도가 낮은 사이트 10개"); print(site_acc.sort_values().head(10).round(3).to_string())
print("[Closed-world] 정확도가 높은 사이트 5개"); print(site_acc.sort_values().tail(5).round(3).to_string())
wrong = pc[pc.label != pc.pred]
pairs = wrong.groupby(["label", "pred"]).size().sort_values(ascending=False).head(15).rename("count").reset_index()
print("[Closed-world] 가장 자주 혼동한 (정답 → 예측) 15쌍"); print(pairs.to_string())
site_acc.to_csv(os.path.join(RESULT_DIR, "svm_closed_site_acc.csv"))
pairs.to_csv(os.path.join(RESULT_DIR, "svm_closed_confused_pairs.csv"), index=False)

[Closed-world] 정확도가 낮은 사이트 10개
label
24    0.700
47    0.700
69    0.700
19    0.725
78    0.725
14    0.725
38    0.750
42    0.750
94    0.750
77    0.750
[Closed-world] 정확도가 높은 사이트 5개
label
43    0.975
26    0.975
12    0.975
75    0.975
88    0.975
[Closed-world] 가장 자주 혼동한 (정답 → 예측) 15쌍
    label  pred  count
0      63    38      3
1      13    40      3
2      10    27      2
3      81    48      2
4      16    89      2
5      80    43      2
6      80     4      2
7      19    22      2
8      78    40      2
9      21    14      2
10     25    24      2
11     27     8      2
12     72    27      2
13     28    57      2
14     69    79      2


## 9. 특징 그룹 제거 실험

LR과 같은 방식: 전체 특징(109개) 기준 + 6개 그룹을 하나씩 뺀 조건 = 7조건을 **같은 분할·같은 C·gamma**로 다시 학습하고 5-fold 정확도 변화를 본다.
SVM은 계수·중요도가 없으므로 그룹 기여도는 이 표로만 판단한다.
- Closed-world: 4번 조건 B의 C·gamma로 고정
- Open-world 이진: 5번 C·gamma는 특징 53개(조건 A)에 맞춘 값이라 109개에 그대로 쓰면 gamma가 너무 커서 모든 기록을 mon으로 예측한다(CV 0.6563 = 다수 범주 비율). 그래서 **109개 기준으로 이진 C·gamma를 먼저 찾고** 그 값으로 고정한다. 탐색 범위는 Closed-world 조건 B 최적값(log2C 6, log2γ −6) 주변이다.

In [16]:
def ablation(df, ycol, fixed, tag):
    tr = df[df.split == "train"]
    rows = []
    for g in [None] + list(GROUPS):
        feats = FEATS_ALL if g is None else [c for c in FEATS_ALL if c not in GROUPS[g]]
        info = tune(tr[feats], tr[ycol], f"{tag} -{g}", fixed=fixed)
        rows.append(dict(removed=g or f"(없음, {len(FEATS_ALL)}개)", n_features=len(feats), cv_acc=info["cv_acc"], cv_std=info["cv_std"]))
    out = pd.DataFrame(rows); out["delta_vs_full"] = out.cv_acc - out.cv_acc.iloc[0]
    return out

if RUN_ABLATION:
    abl = ablation(mon, "label", (closed_rows["B"]["log2C"], closed_rows["B"]["log2gamma"]), "ablation")
    print(abl.round(4).to_string()); abl.to_csv(os.path.join(RESULT_DIR, "svm_ablation_closed.csv"), index=False)
    bv = binary_view(mon, unmon); bv_tr = bv[bv.split == "train"]
    info_b = tune(bv_tr[FEATS_ALL], bv_tr.y, "binary_cv_B", log2C=[4, 6, 8], log2G=[-8, -6, -4], max_extend=1)
    BEST_BINARY_B = (info_b["log2C"], info_b["log2gamma"])
    abl_b = ablation(bv, "y", BEST_BINARY_B, "ablation_bin")
    print(abl_b.round(4).to_string()); abl_b.to_csv(os.path.join(RESULT_DIR, "svm_ablation_binary.csv"), index=False)
else:
    print("RUN_ABLATION = False → 건너뜀")

[ablation -None] best log2C=6, log2γ=-6 | CV acc = 0.8428 ± 0.0051 | 평가한 조합 1개 | 탐색 29s
[ablation -g1] best log2C=6, log2γ=-6 | CV acc = 0.8418 ± 0.0055 | 평가한 조합 1개 | 탐색 26s
[ablation -g2] best log2C=6, log2γ=-6 | CV acc = 0.8336 ± 0.0089 | 평가한 조합 1개 | 탐색 26s
[ablation -g3] best log2C=6, log2γ=-6 | CV acc = 0.8361 ± 0.0038 | 평가한 조합 1개 | 탐색 26s
[ablation -g4] best log2C=6, log2γ=-6 | CV acc = 0.8113 ± 0.0087 | 평가한 조합 1개 | 탐색 29s
[ablation -g5] best log2C=6, log2γ=-6 | CV acc = 0.8230 ± 0.0070 | 평가한 조합 1개 | 탐색 28s
[ablation -g6] best log2C=6, log2γ=-6 | CV acc = 0.7979 ± 0.0051 | 평가한 조합 1개 | 탐색 27s
      removed  n_features  cv_acc  cv_std  delta_vs_full
0  (없음, 109개)         109  0.8428  0.0051         0.0000
1          g1         106  0.8418  0.0055        -0.0010
2          g2          92  0.8336  0.0089        -0.0092
3          g3         103  0.8361  0.0038        -0.0067
4          g4         106  0.8112  0.0087        -0.0316
5          g5          79  0.8230  0.0070        -0.01

## 10. 결과 저장

In [17]:
summary = pd.DataFrame(RESULTS)
cols = ["experiment", "trim", "n_features", "best_C", "best_gamma", "log2C", "log2gamma", "cv_acc", "cv_std", "test_acc",
        "majority_baseline", "mon_as_mon", "unmon_as_unmon", "mon_site_exact", "precision", "fpr", "roc_auc", "pr_auc",
        "test_macro_f1", "TP", "FN", "TN", "FP", "n_support", "search_time_s", "refit_time_s", "predict_time_s"]
summary = summary[[c for c in cols if c in summary.columns]]
print(summary.round(4).to_string())
summary.to_csv(os.path.join(RESULT_DIR, "svm_summary.csv"), index=False)
pd.concat(CV_TABLES.values(), ignore_index=True).to_csv(os.path.join(RESULT_DIR, "svm_cv_tables.csv"), index=False)
for k, v in PREDS.items():
    v.to_csv(os.path.join(RESULT_DIR, f"svm_pred_{k}.csv"), index=False)
print("저장 완료:", sorted(os.listdir(RESULT_DIR)))

             experiment  trim  n_features   best_C best_gamma  log2C  log2gamma  cv_acc  cv_std  test_acc  majority_baseline  mon_as_mon  unmon_as_unmon  mon_site_exact  precision     fpr  roc_auc  pr_auc  test_macro_f1      TP     FN      TN     FP  n_support  search_time_s  refit_time_s  predict_time_s
0          closed_world     0          53  16384.0       0.25   14.0       -2.0  0.8523  0.0057    0.8650                NaN         NaN             NaN             NaN        NaN     NaN      NaN     NaN         0.8646     NaN    NaN     NaN    NaN    10359.0        23.0961        5.6104          5.2225
1    closed_world_condB     0         109     64.0   0.015625    6.0       -6.0  0.8428  0.0051    0.8534                NaN         NaN             NaN             NaN        NaN     NaN      NaN     NaN         0.8538     NaN    NaN     NaN    NaN    12230.0        27.6163        5.8622          7.4340
2  closed_world_default     0          53      1.0      scale    NaN        NaN  0